<a href="https://colab.research.google.com/github/sachitha-m-k/Statistical-Learning-e23168/blob/main/Kalman_filter_assignment_SOLUTIONS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kalman Filter Assignment — Solutions

# Q. Analytical Derivation

## 1. Distribution of $x_k^-$

We are given
$$x^{-}_k = A_{k-1}\,x^{+}_{k-1} + G_{k-1}\,w_{k-1}, \qquad x^{+}_{k-1}\sim\mathscr N(m_{k-1},P_{k-1}),\qquad w_{k-1}\sim\mathscr N(0,\Sigma_p),$$

with $x^+_{k-1}$ and $w_{k-1}$ independent.

**Claim:** A linear combination of jointly independent Gaussian random vectors is itself Gaussian.

*Proof sketch (characteristic function argument).* For any vector $s$,
$$
\mathbb E\!\left[e^{is^T x_k^-}\right]
=\mathbb E\!\left[e^{is^T(A_{k-1}x^+_{k-1}+G_{k-1}w_{k-1})}\right]
=\mathbb E\!\left[e^{i(A_{k-1}^Ts)^Tx^+_{k-1}}\right]\,\mathbb E\!\left[e^{i(G_{k-1}^Ts)^Tw_{k-1}}\right],
$$
where the factorization uses independence of $x_{k-1}^+$ and $w_{k-1}$. Substituting the Gaussian characteristic functions
$$
\mathbb E[e^{iu^Tx_{k-1}^+}]=e^{iu^Tm_{k-1}-\frac12u^TP_{k-1}u},\qquad
\mathbb E[e^{iu^Tw_{k-1}}]=e^{-\frac12u^T\Sigma_p u},
$$
with $u=A_{k-1}^Ts$ and $u=G_{k-1}^Ts$ respectively gives
$$
\mathbb E[e^{is^Tx_k^-}]
=\exp\!\Big(is^TA_{k-1}m_{k-1}-\tfrac12 s^T\big(A_{k-1}P_{k-1}A_{k-1}^T+G_{k-1}\Sigma_pG_{k-1}^T\big)s\Big),
$$
which is exactly the characteristic function of $\mathscr N(m_k^-,P_k^-)$ with
$$
m_k^- = A_{k-1}m_{k-1},\qquad
P_k^- = A_{k-1}P_{k-1}A_{k-1}^T+G_{k-1}\Sigma_pG_{k-1}^T.
$$

*Equivalently (moments argument).* Since $x_k^-$ is an affine function of jointly Gaussian $(x_{k-1}^+,w_{k-1})$, it is Gaussian, and matching the first two moments:
$$
\mathbb E[x_k^-]=A_{k-1}\mathbb E[x_{k-1}^+]+G_{k-1}\mathbb E[w_{k-1}]=A_{k-1}m_{k-1},
$$
$$
\operatorname{Cov}(x_k^-)=A_{k-1}\operatorname{Cov}(x_{k-1}^+)A_{k-1}^T+G_{k-1}\operatorname{Cov}(w_{k-1})G_{k-1}^T
=A_{k-1}P_{k-1}A_{k-1}^T+G_{k-1}\Sigma_pG_{k-1}^T,
$$
where cross terms vanish because $x_{k-1}^+\perp w_{k-1}$. $\blacksquare$

## 2. Distribution of $y_k^-$

Given $y_k^- = H_kx_k^- + z_k$ with $z_k\sim\mathscr N(0,\Sigma_m)$ independent of $x_k^-$, and $x_k^-\sim\mathscr N(m_k^-,P_k^-)$ from Part 1, the same affine-Gaussian argument applies directly: $y_k^-$ is an affine map of jointly Gaussian $(x_k^-,z_k)$, hence Gaussian, with

$$
\mathbb E[y_k^-] = H_k\mathbb E[x_k^-] + \mathbb E[z_k] = H_km_k^-,
$$
$$
\operatorname{Cov}(y_k^-) = H_k\operatorname{Cov}(x_k^-)H_k^T + \operatorname{Cov}(z_k) = H_kP_k^-H_k^T+\Sigma_m,
$$

using independence of $x_k^-$ and $z_k$ to kill the cross-covariance terms. Hence
$$
y_k^-\sim\mathscr N\big(H_km_k^-,\;H_kP_k^-H_k^T+\Sigma_m\big). \qquad\blacksquare
$$

## 3. Joint distribution of $(x_k^-, y_k^-)$

Stack $\begin{bmatrix}x_k^-\\y_k^-\end{bmatrix}=\begin{bmatrix}I\\H_k\end{bmatrix}x_k^- + \begin{bmatrix}0\\I\end{bmatrix}z_k$.

This is an affine function of the jointly Gaussian vector $(x_k^-, z_k)$, so the stacked vector is itself Gaussian (any linear combination of its entries is a linear combination of entries of $x_k^-$ and $z_k$, hence univariate Gaussian — this is the defining property of joint Gaussianity).

The mean follows immediately from Parts 1–2:
$$
\mathbb E\begin{bmatrix}x_k^-\\y_k^-\end{bmatrix}=\begin{bmatrix}m_k^-\\H_km_k^-\end{bmatrix}.
$$

For the covariance we need the three blocks. The $(1,1)$ block is $P_k^-$ and the $(2,2)$ block is $H_kP_k^-H_k^T+\Sigma_m$, both from above. The cross-covariance is
$$
\operatorname{Cov}(x_k^-,y_k^-)=\operatorname{Cov}\big(x_k^-,\,H_kx_k^-+z_k\big)=\operatorname{Cov}(x_k^-,x_k^-)H_k^T+\operatorname{Cov}(x_k^-,z_k)=P_k^-H_k^T,
$$
since $x_k^- \perp z_k$. By symmetry $\operatorname{Cov}(y_k^-,x_k^-)=H_kP_k^-$. Hence
$$
\begin{bmatrix}x_k^- \\ y_k^-\end{bmatrix}\sim\mathscr N\!\left(
\begin{bmatrix}m_k^- \\ H_km_k^-\end{bmatrix},\;
\begin{bmatrix}P_k^- & P_k^-H_k^T\\ H_kP_k^- & H_kP_k^-H_k^T+\Sigma_m\end{bmatrix}
\right). \qquad\blacksquare
$$

## 4. Posterior $x_k^+ = (x_k^- \mid y_k^- = y_k^{\mathrm{obs}})$

This follows from the standard conditioning formula for jointly Gaussian vectors. If
$$
\begin{bmatrix}u\\v\end{bmatrix}\sim\mathscr N\left(\begin{bmatrix}\mu_u\\\mu_v\end{bmatrix},\begin{bmatrix}\Sigma_{uu}&\Sigma_{uv}\\\Sigma_{vu}&\Sigma_{vv}\end{bmatrix}\right),
$$
then, provided $\Sigma_{vv}$ is invertible,
$$
(u\mid v=v_0)\sim\mathscr N\Big(\mu_u+\Sigma_{uv}\Sigma_{vv}^{-1}(v_0-\mu_v),\;\;\Sigma_{uu}-\Sigma_{uv}\Sigma_{vv}^{-1}\Sigma_{vu}\Big).
$$

*(This formula itself follows from completing the square in the joint Gaussian density and recognizing the resulting quadratic form in $u$ as that of a Gaussian density in $u$ conditioned on $v=v_0$; equivalently, it is the standard linear MMSE/affine projection result for jointly Gaussian vectors.)*

Applying this with $u=x_k^-$, $v=y_k^-$, and the moments from Part 3:
$$
\mu_u=m_k^-,\quad \mu_v=H_km_k^-,\quad \Sigma_{uu}=P_k^-,\quad \Sigma_{uv}=P_k^-H_k^T,\quad \Sigma_{vv}=H_kP_k^-H_k^T+\Sigma_m,
$$
we obtain, defining the **Kalman gain**
$$
K_k \triangleq P_k^-H_k^T\big(H_kP_k^-H_k^T+\Sigma_m\big)^{-1},
$$
the posterior mean
$$
m_k = m_k^- + K_k\big(y_k^{\mathrm{obs}} - H_km_k^-\big),
$$
and posterior covariance
$$
P_k = P_k^- - K_kH_kP_k^- = (I-K_kH_k)P_k^-.
$$

Hence $x_k^+ \sim \mathscr N(m_k, P_k)$ with $m_k,P_k$ as above. $\blacksquare$

## 5. Conditional mean and variance

Directly from Part 4 (the conditional distribution *is* $x_k^+$, so its mean and variance are exactly the filter update quantities):

$$
\mathbb E[x_k^-\mid y_k^-=y_k^{\mathrm{obs}}] = m_k = m_k^- + K_k\big(y_k^{\mathrm{obs}}-H_km_k^-\big),
$$

$$
\operatorname{Var}(x_k^-\mid y_k^-=y_k^{\mathrm{obs}}) = P_k = (I-K_kH_k)P_k^-,
$$

where $K_k = P_k^-H_k^T(H_kP_k^-H_k^T+\Sigma_m)^{-1}$.

Note that, unlike for general (non-Gaussian) random vectors, here the conditional variance $P_k$ does **not** depend on the observed value $y_k^{\mathrm{obs}}$ — only on $P_k^-, H_k, \Sigma_m$. This is a special feature of the jointly-Gaussian/linear setting: the Kalman gain and posterior covariance can be precomputed entirely offline, before any data is observed.

# Q. 1-D Example

## 1. Prediction step ($m_k^-, P_k^-$)

This is the scalar specialization of Part 1 of the analytical derivation with $A_{k-1}=a$ (scalar), $G_{k-1}=1$, $\Sigma_p=q$:
$$
x_k^- = a\,x_{k-1}^+ + w_{k-1},\qquad x_{k-1}^+\sim\mathscr N(m_{k-1},P_{k-1}),\quad w_{k-1}\sim\mathscr N(0,q),
$$
with $x_{k-1}^+\perp w_{k-1}$. Since $x_k^-$ is an affine combination of independent Gaussians it is Gaussian, with
$$
\mathbb E[x_k^-] = a\,\mathbb E[x_{k-1}^+] = a\,m_{k-1} = m_k^-,
$$
$$
\operatorname{Var}(x_k^-) = a^2\operatorname{Var}(x_{k-1}^+) + \operatorname{Var}(w_{k-1}) = a^2P_{k-1}+q = P_k^-,
$$
where the variance splits additively because $x_{k-1}^+$ and $w_{k-1}$ are independent (no covariance cross-term). $\blacksquare$

## 2. Update step ($m_k, P_k$)

This is the scalar case of Part 4 above, with $H_k=h$, $\Sigma_m=r$. Define the **innovation**
$$
v_k \triangleq y_k^{\mathrm{obs}} - h\,m_k^-,
$$
and the **innovation variance**
$$
S_k \triangleq h^2P_k^- + r,
$$
so the (scalar) Kalman gain is
$$
K_k = \frac{P_k^-h}{S_k}.
$$
Substituting into the general update formulas $m_k=m_k^-+K_kv_k$ and $P_k=(1-K_kh)P_k^-$ gives exactly
$$
m_k = m_k^- + \frac{P_k^-h}{S_k}\big(y_k^{\mathrm{obs}}-hm_k^-\big),
\qquad
P_k = \Big(1-\frac{P_k^-h^2}{S_k}\Big)P_k^-. \qquad\blacksquare
$$

## 3. Predictive measurement distribution $p(y_k^-\mid Y_{k-1})$

Given only $Y_{k-1}$ (no observation of $y_k$ yet), $x_k^-\mid Y_{k-1}\sim\mathscr N(m_k^-,P_k^-)$ (the prior for step $k$, computed from the filter recursion using only past data). Since
$$
y_k^- = h\,x_k^- + z_k,\qquad z_k\sim\mathscr N(0,r),\quad z_k\perp x_k^-,
$$
this is exactly the scalar case of Part 2 of the analytical derivation:
$$
\mathbb E[y_k^-\mid Y_{k-1}] = h\,m_k^-,\qquad \operatorname{Var}(y_k^-\mid Y_{k-1}) = h^2P_k^-+r.
$$
Hence
$$
p(y_k^-\mid Y_{k-1}) = \mathscr N\big(h\,m_k^-,\; h^2P_k^-+r\big). \qquad\blacksquare
$$
(Note this variance is precisely $S_k$ from Part 2 above — the innovation variance *is* the variance of the one-step-ahead predictive measurement distribution.)

## 4. Posterior-predictive measurement distribution $p(y_k^-\mid Y_k)$

Now condition additionally on $y_k^{\mathrm{obs}}$, i.e. on the full history $Y_k = Y_{k-1}\cup\{y_k^{\mathrm{obs}}\}$. The filtered state satisfies $x_k^+ = (x_k^-\mid y_k^-=y_k^{\mathrm{obs}}) \sim\mathscr N(m_k,P_k)$ from Part 2. Writing the measurement model evaluated at the *updated* state,
$$
y_k^- \mid Y_k \;\;"="\;\; h\,x_k^+ + z_k',\qquad z_k'\sim\mathscr N(0,r),
$$
(here $z_k'$ denotes a fresh, independent copy of the measurement noise — i.e. we are asking for the distribution of a *hypothetical* repeated measurement at time $k$ given everything observed up to and including $y_k^{\mathrm{obs}}$), and applying the same affine-Gaussian argument as in Part 2 of the analytical derivation but with the posterior $(m_k,P_k)$ in place of the prior $(m_k^-,P_k^-)$:
$$
\mathbb E[y_k^-\mid Y_k] = h\,m_k,\qquad \operatorname{Var}(y_k^-\mid Y_k) = h^2P_k + r.
$$
Hence
$$
p(y_k^-\mid Y_k) = \mathscr N\big(h\,m_k,\; h^2P_k+r\big). \qquad\blacksquare
$$

## 5. Numerical example and animation

We pick $a=0.95$ (mildly stable/decaying dynamics), $q=0.5$ (process noise), $h=1$ (direct observation of the state, scaled by 1), and $r=1$ (measurement noise), with prior $x_0\sim\mathscr N(0,2)$. We simulate a true trajectory, generate noisy observations, run the scalar Kalman filter recursion, and animate the prior $\mathscr N(m_k^-,P_k^-)$ together with the posterior $\mathscr N(m_k,P_k)$ at each time step, with a vertical line marking the observed value $y_k^{\mathrm{obs}}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.stats import norm
from IPython.display import HTML

np.random.seed(42)

# ----- scalar model parameters -----
a = 0.95   # state transition coefficient
q = 0.5    # process noise variance
h = 1.0    # measurement coefficient
r = 1.0    # measurement noise variance

m0, P0 = 0.0, 2.0   # prior on x_0
N = 30               # number of time steps

# ----- simulate ground truth and noisy measurements -----
x_true = np.zeros(N + 1)
x_true[0] = np.random.normal(m0, np.sqrt(P0))
for k in range(1, N + 1):
    x_true[k] = a * x_true[k - 1] + np.random.normal(0, np.sqrt(q))

y_obs = h * x_true + np.random.normal(0, np.sqrt(r), size=N + 1)

# ----- Kalman filter recursion -----
m_pred = np.zeros(N + 1)   # m_k^-
P_pred = np.zeros(N + 1)   # P_k^-
m_filt = np.zeros(N + 1)   # m_k
P_filt = np.zeros(N + 1)   # P_k

m_filt[0], P_filt[0] = m0, P0
m_pred[0], P_pred[0] = m0, P0   # no prediction step before k=0; treat prior as both

for k in range(1, N + 1):
    # --- predict ---
    m_pred[k] = a * m_filt[k - 1]
    P_pred[k] = a**2 * P_filt[k - 1] + q

    # --- update ---
    S_k = h**2 * P_pred[k] + r          # innovation variance
    K_k = P_pred[k] * h / S_k           # Kalman gain
    v_k = y_obs[k] - h * m_pred[k]       # innovation

    m_filt[k] = m_pred[k] + K_k * v_k
    P_filt[k] = (1 - K_k * h) * P_pred[k]

print(f"{'k':>3} {'m_pred':>8} {'P_pred':>8} {'m_filt':>8} {'P_filt':>8} {'y_obs':>8} {'x_true':>8}")
for k in range(0, N + 1, 5):
    print(f"{k:3d} {m_pred[k]:8.3f} {P_pred[k]:8.3f} {m_filt[k]:8.3f} {P_filt[k]:8.3f} {y_obs[k]:8.3f} {x_true[k]:8.3f}")

In [ ]:
# ----- animate prior vs posterior density at each time step -----
fig, ax = plt.subplots(figsize=(8, 5))

x_min = min((m_pred - 4*np.sqrt(P_pred)).min(), (m_filt - 4*np.sqrt(P_filt)).min())
x_max = max((m_pred + 4*np.sqrt(P_pred)).max(), (m_filt + 4*np.sqrt(P_filt)).max())
xs = np.linspace(x_min, x_max, 400)

def update(k):
    ax.clear()
    prior_pdf = norm.pdf(xs, m_pred[k], np.sqrt(P_pred[k]))
    post_pdf = norm.pdf(xs, m_filt[k], np.sqrt(P_filt[k]))
    ax.plot(xs, prior_pdf, color="tab:blue", lw=2, label=r"prior $\mathcal{N}(m_k^-, P_k^-)$")
    ax.plot(xs, post_pdf, color="tab:orange", lw=2, label=r"posterior $\mathcal{N}(m_k, P_k)$")
    ax.axvline(y_obs[k], color="black", ls="--", lw=1.5, label=r"$y_k^{\mathrm{obs}}$")
    ax.axvline(x_true[k], color="green", ls=":", lw=1.5, label=r"true $x_k$")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(0, 1.0)
    ax.set_title(f"Step k = {k}")
    ax.set_xlabel("x")
    ax.set_ylabel("density")
    ax.legend(loc="upper right", fontsize=9)
    return ax.get_children()

anim = animation.FuncAnimation(fig, update, frames=N + 1, interval=400, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
# Static overview: predicted vs filtered mean trajectories against ground truth
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].plot(x_true, label="true state $x_k$", color="green")
axes[0].plot(y_obs, "k.", alpha=0.5, label=r"observations $y_k^{\mathrm{obs}}$")
axes[0].plot(m_pred, "--", color="tab:blue", label=r"predicted mean $m_k^-$")
axes[0].plot(m_filt, color="tab:orange", label=r"filtered mean $m_k$")
axes[0].legend()
axes[0].set_ylabel("x")
axes[0].set_title("Scalar Kalman filter: state estimate vs. truth and observations")

axes[1].plot(P_pred, "--", color="tab:blue", label=r"predicted variance $P_k^-$")
axes[1].plot(P_filt, color="tab:orange", label=r"filtered variance $P_k$")
axes[1].set_xlabel("time step k")
axes[1].set_ylabel("variance")
axes[1].legend()

plt.tight_layout()
plt.show()

**Observation.** The filtered variance $P_k$ is always smaller than the predicted variance $P_k^-$ at the same step (incorporating a measurement always reduces uncertainty, since $P_k=(1-K_kh)P_k^-$ with $0<K_kh<1$ here), and both variances converge to a steady-state value as $k\to\infty$ because $a,q,h,r$ are time-invariant. The filtered mean $m_k$ tracks the true trajectory $x_k$ much more closely than the raw noisy observations $y_k^{\mathrm{obs}}$.

# Q. 2D-Position Estimation

## Part A

**Setup.** The state is the constant-velocity kinematic vector $x_k=[p_x,p_y,v_x,v_y]^T$, and over a time interval $\Delta t$, in the absence of process noise, basic kinematics give
$$
p_x(k) = p_x(k-1) + \Delta t\, v_x(k-1),\qquad
p_y(k) = p_y(k-1) + \Delta t\, v_y(k-1),
$$
$$
v_x(k) = v_x(k-1),\qquad v_y(k) = v_y(k-1),
$$
i.e. positions update by (velocity $\times$ time) and velocities are unchanged (zero acceleration, "constant velocity" model). Writing this in matrix form $x_k^- = Ax_{k-1}^+$:
$$
\begin{bmatrix}p_x(k)\\p_y(k)\\v_x(k)\\v_y(k)\end{bmatrix}
=
\begin{bmatrix}
1 & 0 & \Delta t & 0\\
0 & 1 & 0 & \Delta t\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}p_x(k-1)\\p_y(k-1)\\v_x(k-1)\\v_y(k-1)\end{bmatrix}
\quad\Longrightarrow\quad
A=\begin{bmatrix}
1 & 0 & \Delta t & 0\\
0 & 1 & 0 & \Delta t\\
0 & 0 & 1 & 0\\
0 & 0 & 0 & 1
\end{bmatrix}. \qquad\blacksquare
$$

**Measurement matrix $H$.** Since only position is measured,
$$
y_k = \begin{bmatrix}p_x^{\mathrm{meas}}(k)\\p_y^{\mathrm{meas}}(k)\end{bmatrix}
= \begin{bmatrix}1&0&0&0\\0&1&0&0\end{bmatrix}
\begin{bmatrix}p_x(k)\\p_y(k)\\v_x(k)\\v_y(k)\end{bmatrix} + z_k
\quad\Longrightarrow\quad
H = \begin{bmatrix}1&0&0&0\\0&1&0&0\end{bmatrix},
$$
which simply selects out the first two (position) components of the state and discards the velocity components. $\blacksquare$

**Process noise input matrix $G$.** The process noise represents an unmodelled, approximately-constant random acceleration $a_w=[a_{w,x},a_{w,y}]^T\sim\mathscr N(0,\Sigma_p)$ acting over the interval $[k-1,k)$. Standard kinematics under constant acceleration over a small interval $\Delta t$ gives the position and velocity increments
$$
\Delta p = \tfrac12 a_w\,\Delta t^2,\qquad \Delta v = a_w\,\Delta t,
$$
(these are exactly the usual $s=\tfrac12at^2$, $v=at$ kinematic relations applied to the random forcing $a_w$ in place of a deterministic acceleration). Stacking these increments for the $x$ and $y$ components into $x_k^- = Ax_{k-1}^+ + Gw_{k-1}$ with $w_{k-1}=a_w$:
$$
G\,w_{k-1}=
\begin{bmatrix}\tfrac12\Delta t^2\, a_{w,x}\\ \tfrac12\Delta t^2\, a_{w,y}\\ \Delta t\, a_{w,x}\\ \Delta t\, a_{w,y}\end{bmatrix}
\quad\Longrightarrow\quad
G=\begin{bmatrix}
\tfrac12\Delta t^2 & 0\\
0 & \tfrac12\Delta t^2\\
\Delta t & 0\\
0 & \Delta t
\end{bmatrix}. \qquad\blacksquare
$$

With $A$, $H$, $G$ as above and $w_{k-1}\sim\mathscr N(0,\Sigma_p)$, $z_k\sim\mathscr N(0,\Sigma_m)$ mutually independent and independent of the initial state, the model
$$
x_k^- = Ax_{k-1}^+ + Gw_{k-1},\qquad y_k^- = Hx_k^- + z_k,
$$
is exactly of the general linear-Gaussian form analyzed in the first question, so the full Kalman filter prediction/update recursion derived there (Parts 1–4) applies directly, with these specific $A,H,G$ plugged in.

## Part B

We now implement the 2-D constant-velocity Kalman filter in Python and apply it to a simulated noisy-GPS trajectory. Since real GPS ground truth isn't available here, we generate a synthetic "true" trajectory (a vehicle moving with slowly varying heading), corrupt it with i.i.d. Gaussian position noise to mimic noisy GPS fixes, and then recover a smoothed trajectory with the Kalman filter, comparing against both the raw measurements and the (synthetic) ground truth.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# ---------------------------------------------------------------
# Model matrices (Part A)
# ---------------------------------------------------------------
dt = 1.0   # time step between GPS fixes, in seconds

A = np.array([
    [1, 0, dt, 0],
    [0, 1, 0, dt],
    [0, 0, 1,  0],
    [0, 0, 0,  1]
])

H = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
])

G = np.array([
    [0.5*dt**2, 0],
    [0, 0.5*dt**2],
    [dt, 0],
    [0, dt]
])

sigma_p = 0.05    # std of random acceleration disturbance (process noise), m/s^2
sigma_m = 3.0     # std of GPS position noise, meters (typical consumer GPS ~ a few metres)

Sigma_p = sigma_p**2 * np.eye(2)
Sigma_m = sigma_m**2 * np.eye(2)

In [ ]:
# ---------------------------------------------------------------
# Simulate a "true" trajectory and noisy GPS measurements
# ---------------------------------------------------------------
N = 60  # number of GPS fixes

x_true = np.zeros((N + 1, 4))
x_true[0] = [0.0, 0.0, 1.0, 0.5]   # start at origin, initial velocity (1, 0.5) m/s

turn_rate = 0.03   # rad per step -- simulates a vehicle slowly turning

for k in range(1, N + 1):
    vx, vy = x_true[k-1, 2], x_true[k-1, 3]
    # rotate the velocity vector slightly to emulate a turning path
    vx_new = vx*np.cos(turn_rate) - vy*np.sin(turn_rate)
    vy_new = vx*np.sin(turn_rate) + vy*np.cos(turn_rate)

    w = np.random.multivariate_normal(np.zeros(2), Sigma_p)  # true random accel. disturbance
    x_true[k, 2] = vx_new
    x_true[k, 3] = vy_new
    x_true[k, 0] = x_true[k-1, 0] + dt*vx_new + G[0] @ w
    x_true[k, 1] = x_true[k-1, 1] + dt*vy_new + G[1] @ w

# Noisy GPS-like measurements of position only
y_obs = np.zeros((N + 1, 2))
for k in range(N + 1):
    z = np.random.multivariate_normal(np.zeros(2), Sigma_m)
    y_obs[k] = H @ x_true[k] + z

print("Simulated", N + 1, "GPS fixes with measurement noise std =", sigma_m, "m")

In [ ]:
# ---------------------------------------------------------------
# Kalman filter implementation
# ---------------------------------------------------------------
def kalman_filter(y_obs, A, H, G, Sigma_p, Sigma_m, m0, P0):
    """
    Run the linear-Gaussian Kalman filter (prediction + update) over a
    sequence of measurements.

    Parameters
    ----------
    y_obs : (N+1, d_y) array of measurements y_0, ..., y_N
    A, H, G : model matrices
    Sigma_p, Sigma_m : process / measurement noise covariances
    m0, P0 : prior mean and covariance of the initial state

    Returns
    -------
    m_filt : (N+1, d_x) filtered means  m_k
    P_filt : (N+1, d_x, d_x) filtered covariances P_k
    m_pred : (N+1, d_x) predicted means m_k^-  (m_pred[0] := m0)
    P_pred : (N+1, d_x, d_x) predicted covariances P_k^- (P_pred[0] := P0)
    """
    N = y_obs.shape[0] - 1
    d_x = A.shape[0]

    m_filt = np.zeros((N + 1, d_x))
    P_filt = np.zeros((N + 1, d_x, d_x))
    m_pred = np.zeros((N + 1, d_x))
    P_pred = np.zeros((N + 1, d_x, d_x))

    I = np.eye(d_x)

    # step 0: no prediction yet, just incorporate the prior and first measurement
    m_pred[0], P_pred[0] = m0, P0
    S0 = H @ P0 @ H.T + Sigma_m
    K0 = P0 @ H.T @ np.linalg.inv(S0)
    m_filt[0] = m0 + K0 @ (y_obs[0] - H @ m0)
    P_filt[0] = (I - K0 @ H) @ P0

    for k in range(1, N + 1):
        # --- predict ---
        m_pred[k] = A @ m_filt[k-1]
        P_pred[k] = A @ P_filt[k-1] @ A.T + G @ Sigma_p @ G.T

        # --- update ---
        S_k = H @ P_pred[k] @ H.T + Sigma_m
        K_k = P_pred[k] @ H.T @ np.linalg.inv(S_k)
        v_k = y_obs[k] - H @ m_pred[k]

        m_filt[k] = m_pred[k] + K_k @ v_k
        P_filt[k] = (I - K_k @ H) @ P_pred[k]

    return m_filt, P_filt, m_pred, P_pred

In [ ]:
# ---------------------------------------------------------------
# Apply the filter to the noisy GPS trajectory
# ---------------------------------------------------------------
m0 = np.array([y_obs[0, 0], y_obs[0, 1], 0.0, 0.0])   # initialize velocity at zero
P0 = np.diag([sigma_m**2, sigma_m**2, 1.0, 1.0])       # uncertain about initial velocity

m_filt, P_filt, m_pred, P_pred = kalman_filter(y_obs, A, H, G, Sigma_p, Sigma_m, m0, P0)

rmse_raw = np.sqrt(np.mean(np.sum((y_obs - x_true[:, :2])**2, axis=1)))
rmse_filt = np.sqrt(np.mean(np.sum((m_filt[:, :2] - x_true[:, :2])**2, axis=1)))

print(f"RMSE of raw noisy GPS positions    : {rmse_raw:6.3f} m")
print(f"RMSE of Kalman-filtered positions  : {rmse_filt:6.3f} m")
print(f"Improvement factor                 : {rmse_raw/rmse_filt:6.2f}x")

In [ ]:
# ---------------------------------------------------------------
# Visualize: true path, noisy GPS fixes, and filtered estimate
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.plot(x_true[:, 0], x_true[:, 1], color="green", lw=2, label="true trajectory")
ax.scatter(y_obs[:, 0], y_obs[:, 1], color="red", s=15, alpha=0.6, label="noisy GPS fixes")
ax.plot(m_filt[:, 0], m_filt[:, 1], color="blue", lw=2, label="Kalman filter estimate")
ax.set_xlabel("$p_x$ (m)")
ax.set_ylabel("$p_y$ (m)")
ax.set_title("2D position tracking: true path vs. noisy GPS vs. KF estimate")
ax.legend()
ax.axis("equal")

ax = axes[1]
time = np.arange(N + 1) * dt
pos_err_raw = np.linalg.norm(y_obs - x_true[:, :2], axis=1)
pos_err_filt = np.linalg.norm(m_filt[:, :2] - x_true[:, :2], axis=1)
ax.plot(time, pos_err_raw, color="red", alpha=0.7, label="raw GPS error")
ax.plot(time, pos_err_filt, color="blue", label="KF estimate error")
ax.set_xlabel("time (s)")
ax.set_ylabel("position error (m)")
ax.set_title("Position error over time")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Bonus: recovered velocity estimates (not directly observed!)
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(time, x_true[:, 2], color="green", ls="-", label=r"true $v_x$")
ax.plot(time, x_true[:, 3], color="darkgreen", ls="-", label=r"true $v_y$")
ax.plot(time, m_filt[:, 2], color="blue", ls="--", label=r"estimated $v_x$")
ax.plot(time, m_filt[:, 3], color="navy", ls="--", label=r"estimated $v_y$")
ax.set_xlabel("time (s)")
ax.set_ylabel("velocity (m/s)")
ax.set_title("Velocity recovered by the Kalman filter\n(velocity is never directly measured -- only inferred from position GPS fixes)")
ax.legend()
plt.tight_layout()
plt.show()

**Discussion.**
- The Kalman filter substantially reduces position error relative to the raw noisy GPS measurements (typically by a factor of $1.5$–$2\times$ in RMSE for this noise level), because it fuses the noisy measurement with a dynamical prediction based on the constant-velocity model, effectively averaging out measurement noise while still tracking real motion.
- Even though velocity is **never directly measured**, the filter recovers a smooth and reasonably accurate estimate of $(v_x,v_y)$ purely from the structure imposed by the state-transition matrix $A$ and the sequence of noisy position fixes — this is one of the most useful properties of the Kalman filter for sensor fusion problems.
- The process noise level $\sigma_p$ controls how much the filter "trusts" the constant-velocity model vs. the incoming measurements: smaller $\sigma_p$ produces a smoother but more sluggish (higher-lag) estimate that struggles to track genuine turns; larger $\sigma_p$ tracks turns faster but is noisier, since the filter then puts more weight on each new GPS fix relative to its own prediction. Try changing `sigma_p` and `sigma_m` above and re-running the cells to see this trade-off directly.